In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sea
import scipy as sp
%matplotlib inline

In [58]:
pd.set_option('display.max_columns', None)

## Load Dataset

In [59]:
dataset = pd.read_csv("ipl_deliveries.csv")

In [60]:
dataset.head()

,Unnamed: 0,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,wide_runs,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN
1,1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN
2,2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,0,0,0,0,0,4,0,4,NaN,NaN,NaN
3,3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN
4,4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,2,0,0,0,0,0,2,2,NaN,NaN,NaN


In [61]:
dataset.isna().sum()
delivery_dataset = dataset.copy()

### Filling Missing values

In [62]:
### Filling in missing values: in player_dismissed, dismissal_kind, fielder.

delivery_dataset['player_dismissed'] = delivery_dataset['player_dismissed'].fillna('not out')

delivery_dataset['dismissal_kind'] = delivery_dataset['dismissal_kind'].fillna('N/A')

delivery_dataset['fielder'] = delivery_dataset['fielder'].fillna('N/A')

### Dropping Duplicates

In [63]:
# display(delivery_dataset.loc[delivery_dataset.duplicated().sum()])
print(f"Number of duplicated rows before: {delivery_dataset.duplicated().sum()}")

print(f"Number of duplicated rows after: {delivery_dataset.duplicated().sum()}")

Number of duplicated rows before: 0
Number of duplicated rows after: 0


In [64]:
delivery_dataset.head()

,Unnamed: 0,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,wide_runs,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,0,0,0,0,0,0,0,0,not out,N/A,N/A
1,1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,0,0,0,0,0,0,0,0,not out,N/A,N/A
2,2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,0,0,0,0,0,4,0,4,not out,N/A,N/A
3,3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,0,0,0,0,0,0,0,0,not out,N/A,N/A
4,4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,2,0,0,0,0,0,2,2,not out,N/A,N/A


### Standardising Team Names

In [65]:
all_teams = np.unique(delivery_dataset['batting_team'])

print(f"Teams before standardising names: {len(all_teams)}")
display(all_teams)

same_team_names = {
    'Rising Pune Supergiants' : 'Rising Pune Supergiant',
    'Delhi Daredevils' : 'Delhi Capitals',
    'Deccan Chargers' : 'Sunrisers Hyderabad',
    'Kings XI Punjab' : 'Punjab Kings'
}

## Function to standardised team name:
def standardised_team_names(df,columns):
    for col in columns:
        df[col] = df[col].replace(same_team_names)
    return df

team_columns = ['batting_team', 'bowling_team']
delivery_dataset = standardised_team_names(delivery_dataset, team_columns)

all_teams = np.unique(delivery_dataset['batting_team'])
print(f"Teams after standardising names: {len(all_teams)}")
display(all_teams)

Teams before standardising names: 14


array(['Chennai Super Kings', 'Deccan Chargers', 'Delhi Daredevils',
       'Gujarat Lions', 'Kings XI Punjab', 'Kochi Tuskers Kerala',
       'Kolkata Knight Riders', 'Mumbai Indians', 'Pune Warriors',
       'Rajasthan Royals', 'Rising Pune Supergiant',
       'Rising Pune Supergiants', 'Royal Challengers Bangalore',
       'Sunrisers Hyderabad'], dtype=object)

Teams after standardising names: 12


array(['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Lions',
       'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Mumbai Indians',
       'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals',
       'Rising Pune Supergiant', 'Royal Challengers Bangalore',
       'Sunrisers Hyderabad'], dtype=object)

### Finding the batsman stats first:

Top socring batsman, most dismissed batsman, batsman total balls, 
strike rate, total_dismissals, batting average, batsman fours and sixes,
boundary %tage, runs per match, count hundreds, count fifties, strike rate. 

In [66]:
batsman_team = delivery_dataset.groupby('batsman')['batting_team'].unique()

In [68]:
# Finding the batsman stats first:
# batsman total balls balls
batsman_total_balls = delivery_dataset['batsman'].value_counts().reset_index()
batsman_total_balls.rename(columns={'count':'no.of_balls'}, inplace=True)


# batsman runs
batsman_runs = delivery_dataset.groupby('batsman')['batsman_runs'].sum().sort_values(ascending=False).reset_index()


#making a new df for batsman and will use this for eda!
batsman_stats = pd.merge(batsman_runs, batsman_total_balls, on='batsman', how='left')


# batsman dismissals
batsman_dismissals = delivery_dataset['player_dismissed'].value_counts().reset_index()
batsman_dismissals.columns = ['batsman', 'dismissals_count']
batsman_dismissals = batsman_dismissals[batsman_dismissals['batsman'] != 'not out']
# merging:
batsman_stats = pd.merge(batsman_stats, batsman_dismissals, on='batsman', how='left')
batsman_stats['dismissals_count'] = batsman_stats['dismissals_count'].fillna(0)


#strike rate
batsman_stats['strike_rate'] = (batsman_stats['batsman_runs'] / batsman_stats['no.of_balls'].replace(0,np.nan)) * 100
batsman_stats['strike_rate'] = batsman_stats['strike_rate'].fillna(np.inf)


# batting average
batsman_stats['batting_average'] = batsman_stats['batsman_runs'] / batsman_stats['dismissals_count'].replace(0,np.nan)
batsman_stats['batting_average'] = batsman_stats['batting_average'].fillna(np.inf)


# batsman and played teams
batsman_stats = pd.merge(batsman_stats, batsman_team, on='batsman', how='left')


# bstsman fours and sixes.
batsman_fours = delivery_dataset[delivery_dataset['batsman_runs'] == 4].groupby('batsman')['batsman_runs'].count().reset_index()
batsman_fours.rename(columns={'batsman_runs':'no_of_fours'}, inplace=True)

batsman_sixes= delivery_dataset[delivery_dataset['batsman_runs'] == 6].groupby('batsman')['batsman_runs'].count().reset_index()
batsman_sixes.rename(columns={'batsman_runs':'no_of_sixes'}, inplace=True)

batsman_stats = pd.merge(batsman_stats, batsman_fours, on='batsman', how='left')
batsman_stats = pd.merge(batsman_stats, batsman_sixes, on='batsman', how='left')

batsman_stats['no_of_fours'] = batsman_stats['no_of_fours'].fillna(0).astype(int)
batsman_stats['no_of_sixes'] = batsman_stats['no_of_sixes'].fillna(0).astype(int)


# boundary percentage
batsman_stats['boundary_percentage'] = (
    ((batsman_stats['no_of_fours'] * 4) + (batsman_stats['no_of_sixes'] * 6)) / batsman_stats['batsman_runs'].replace(0,np.nan)) * 100

batsman_stats['boundary_percentage'] = batsman_stats['boundary_percentage'].fillna(0)


##batsman_runs_per_match
batsman_runs_per_match = delivery_dataset.groupby(['match_id','batsman'])['batsman_runs'].sum().reset_index()

def count_hundreds(runs):
    return (runs>=100).sum()

def count_fifties(runs):
    return (runs>=50).sum()

hundreds = batsman_runs_per_match.groupby('batsman')['batsman_runs'].apply(count_hundreds).reset_index(name='no_of_100s')
fifties = batsman_runs_per_match.groupby('batsman')['batsman_runs'].apply(count_fifties).reset_index(name='no_of_50s')

batsman_stats = pd.merge(batsman_stats, hundreds, on='batsman', how='left')
batsman_stats = pd.merge(batsman_stats, fifties, on='batsman', how='left')

batsman_stats['no_of_100s']= batsman_stats['no_of_100s'].fillna(0).astype(int)
batsman_stats['no_of_50s']= batsman_stats['no_of_50s'].fillna(0).astype(int)

In [69]:
batsman_stats

,batsman,batsman_runs,no.of_balls,dismissals_count,strike_rate,batting_average,batting_team,no_of_fours,no_of_sixes,boundary_percentage,no_of_100s,no_of_50s
0,SK Raina,4548,3369,134.0,134.995548,33.940299,"[Gujarat Lions, Chennai Super Kings]",402,174,58.311346,1,32
1,V Kohli,4423,3494,118.0,126.588437,37.483051,[Royal Challengers Bangalore],384,160,56.432286,4,35
2,RG Sharma,4207,3274,129.0,128.497251,32.612403,"[Mumbai Indians, Sunrisers Hyderabad]",354,173,58.331353,1,33
3,G Gambhir,4132,3433,131.0,120.361200,31.541985,"[Kolkata Knight Riders, Delhi Capitals]",484,58,55.275895,0,35
4,DA Warner,4014,2902,100.0,138.318401,40.140000,"[Sunrisers Hyderabad, Delhi Capitals]",401,160,63.876432,3,39
...,...,...,...,...,...,...,...,...,...,...,...,...
456,Abdur Razzak,0,2,0.0,0.000000,inf,[Royal Challengers Bangalore],0,0,0.000000,0,0
457,S Kaushik,0,1,1.0,0.000000,0.000000,[Gujarat Lions],0,0,0.000000,0,0
458,S Ladda,0,10,1.0,0.000000,0.000000,[Delhi Capitals],0,0,0.000000,0,0
459,U Kaul,0,1,0.0,0.000000,inf,[Punjab Kings],0,0,0.000000,0,0


Finding the bowler stats:
bowler who bowled most balls, most run giving bowler(conceeded), most_wide, most_no balls, bye runs, legbye runs, penalty runs, extra runs overall.
 bowler credited dismisalls, wicket by bowlers, bpowling avg, matches_played by bowler,no.of balls bowled, total_overs, dot balls, wickets pers match, 4 wicket haul and 5 wicket haul, 
bowling_economy, avg dot balls, bowling_SR, 


In [78]:
#balls bowled by bowler
total_balls_bowled = delivery_dataset['bowler'].value_counts().reset_index()

#total runs conceeded
total_runs_conceeded_by_bowler = delivery_dataset.groupby('bowler')['total_runs'].sum().sort_values(ascending=False).reset_index()
total_runs_conceeded_by_bowler.rename(columns={'total_runs':'runs_conceded'}, inplace=True)

# most wides
bowled_most_wide = delivery_dataset.groupby('bowler')['wide_runs'].sum().sort_values(ascending= False)

# most no balls
bowled_most_no_balls = delivery_dataset.groupby('bowler')['noball_runs'].sum().sort_values(ascending= False)

#most bye runs
bowled_most_by_runs = delivery_dataset.groupby('bowler')['bye_runs'].sum().sort_values(ascending= False)

# most leg bye
bowled_most_leg_bye = delivery_dataset.groupby('bowler')['legbye_runs'].sum().sort_values(ascending= False)

# most penalty runs
bowled_most_penalty_runs = delivery_dataset.groupby('bowler')['penalty_runs'].sum().sort_values(ascending=False)

# msot extra runs overall
bowled_extra_runs = delivery_dataset.groupby('bowler')['extra_runs'].sum().sort_values(ascending=False)

#bowler credited wickets
bowler_credited_dismissal = delivery_dataset[
    delivery_dataset['dismissal_kind'].isin(['caught', 'bowled', 'lbw' ,'stumped', 'caught and bowled'])
]

wickets_by_bowler = bowler_credited_dismissal.groupby('bowler')['dismissal_kind'].count().sort_values(ascending=False)
wickets_by_bowler = wickets_by_bowler.reset_index()
wickets_by_bowler.rename(columns= {'dismissal_kind':'no_of_wickets'}, inplace=True)

# Bowler stats: bowling_avg, totalmatches, total overs, total dot balls, wicket per match (matchid), four wickets, five wickets, total matches played, 
# bowling economy, avg  dot balls, bowling SR 
